# 01 — Tiền xử lý ảnh làn đường nâng cao & Phân loại làn đường Vương Quốc Anh (UK Left-Hand Traffic)

**Mục tiêu của Notebook này:**
1. Cải tiến pipeline tiền xử lý ảnh từ `data/raw/` đạt chuẩn chất lượng cho bài toán tự lái trong **Forza Horizon 4 (Bản đồ Vương Quốc Anh - Lái xe bên trái)**:
   - **Bước 1:** Cân bằng sáng cục bộ **CLAHE** trên kênh L (không gian màu LAB) + khử nhiễu **Gaussian Blur**.
   - **Bước 2:** Kết hợp phát hiện biên **Canny** và **HLS Color Mask** (lọc vạch kẻ đường trắng & vàng).
   - **Bước 3:** Cắt vùng quan tâm **ROI 다 giác (Polygon ROI)** nhìn xa tới chân trời để bắt góc cua sớm.
   - **Bước 4:** Lọc nhiễu đốm nhỏ (**Connected Components Blob Filter**) + **Morphological Closing/Dilation**.
   - **Bước 5:** Phát hiện đoạn thẳng **Probabilistic Hough Transform**, lọc độ dốc, nội suy đường cong **B-Spline** mượt.
   - **Bước 6:** **Phân loại làn đường chuẩn UK (Left-Hand Traffic)**: xác định biên trái lề đường (`uk_left_edge`), vạch phân cách giữa đường (`uk_center_divider`), biên phải làn đối diện (`uk_right_opposite_edge`), và đường dẫn tâm làn trái (`uk_left_lane`).
2. Xuất toàn bộ ảnh đã xử lý và bộ dữ liệu chuẩn YOLO11-seg (`train/`, `valid/`, `highlighted/`) vào thư mục **`data/processing/`**.
3. Lưu bảng số liệu **`preprocessing_metrics.csv`** và biểu đồ **`preprocessing_pipeline_chart.png`** trực tiếp tại thư mục gốc (**Root**) của dự án.

In [1]:
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tìm thư mục gốc của dự án
ROOT_DIR = Path.cwd().parent if (Path.cwd().parent / "README.md").exists() else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.path import RAW_DATA_DIR, PROCESSING_DATA_DIR
from src.preprocessing.batch_processor import (
    preprocess_single_road_image,
    run_batch_preprocessing,
)

print(f"Project Root        : {ROOT_DIR}")
print(f"Raw Data Directory  : {RAW_DATA_DIR}")
print(f"Processing Directory: {PROCESSING_DATA_DIR}")

Project Root        : h:\FPT\4\CPV\project\CPV301
Raw Data Directory  : h:\FPT\4\CPV\project\CPV301\data\raw
Processing Directory: h:\FPT\4\CPV\project\CPV301\data\processing


## Phần 1 — Kiểm thử & Trực quan hóa từng bước Tiền xử lý trên ảnh mẫu

In [2]:
# Tìm danh sách ảnh trong data/raw
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_files = sorted([
    p for p in Path(RAW_DATA_DIR).rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
])

assert image_files, f"Không tìm thấy ảnh trong {RAW_DATA_DIR}"
sample_idx = min(9, len(image_files) - 1)
image_path = image_files[sample_idx]
image = cv2.imread(str(image_path))

print(f"Tổng số ảnh trong data/raw: {len(image_files)}")
print(f"Ảnh mẫu đang chọn        : {image_path}")
print(f"Kích thước ảnh           : {image.shape[1]} x {image.shape[0]} px")

Tổng số ảnh trong data/raw: 134571
Ảnh mẫu đang chọn        : h:\FPT\4\CPV\project\CPV301\data\raw\TUSimple\test_set\clips\0530\1492626047222176976_0\15.jpg
Kích thước ảnh           : 1280 x 720 px


In [3]:
# Chạy pipeline 6 bước trên ảnh mẫu
stages = preprocess_single_road_image(image)

print("=== THÔNG SỐ PHÂN TÍCH LÀN ĐƯỜNG UK TRÊN ẢNH MẪU ===")
print(f"- Số đoạn thẳng Hough phát hiện : {stages['num_hough_segments']}")
print(f"- Số làn đường B-Spline hội tụ  : {stages['num_detected_lanes']}")
print(f"- Chế độ bám làn UK             : {stages['uk_lane_type']}")
print(f"- Tỷ lệ pixel biên trong ROI    : {stages['edge_Validate_ratio']:.4f}")
print(f"- Tỷ lệ pixel vạch trắng/vàng   : {stages['white_mask_ratio']:.4f}")

# Hiển thị trực quan 6 bước xử lý
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
panels = [
    (cv2.cvtColor(stages["original"], cv2.COLOR_BGR2RGB), "1. Ảnh gốc (Original BGR)", None),
    (stages["enhanced_gray"], "2. Cân bằng sáng CLAHE + Khử nhiễu", "gray"),
    (stages["roi_edges"], "3. Biên Canny + HLS trong vùng ROI", "gray"),
    (stages["clean_edges"], "4. Lọc nhiễu Blob + Làm dày hình thái học", "gray"),
    (stages["color_mask"], "5. Mặt nạ màu vạch đường (Trắng/Vàng)", "gray"),
    (cv2.cvtColor(stages["highlighted"], cv2.COLOR_BGR2RGB), f"6. Kết quả B-Spline & Làn UK ({stages['uk_lane_type']})", None),
]

for ax, (panel_img, title, cmap) in zip(axes.flat, panels):
    ax.imshow(panel_img, cmap=cmap)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle("Pipeline Tiền xử lý Làn đường CPV301 (Chuẩn Giao thông Vương Quốc Anh - LHT)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

=== THÔNG SỐ PHÂN TÍCH LÀN ĐƯỜNG UK TRÊN ẢNH MẪU ===
- Số đoạn thẳng Hough phát hiện : 217
- Số làn đường B-Spline hội tụ  : 2
- Chế độ bám làn UK             : uk_standard_left_lane
- Tỷ lệ pixel biên trong ROI    : 0.0152
- Tỷ lệ pixel vạch trắng/vàng   : 0.0104


C:\Users\pc\AppData\Local\Temp\ipykernel_4220\609476675.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Phần 2 — Xử lý hàng loạt (Batch Processing) và Xuất dữ liệu ra `data/processing/` + Root

Cell dưới đây thực hiện:
1. Chạy pipeline tiền xử lý nâng cao trên tập ảnh `data/raw/` và lưu ảnh kết quả vào **`data/processing/highlighted/`**.
2. Chuyển đổi dữ liệu TuSimple kèm tiền xử lý CLAHE sang định dạng **YOLO11-seg** (`data/processing/train/` và `data/processing/valid/`) để phục vụ trực tiếp cho `02_train.ipynb`.
3. Xuất file số liệu **`preprocessing_metrics.csv`** và biểu đồ **`preprocessing_pipeline_chart.png`** ra thư mục gốc (`ROOT_DIR`) và `data/processing/`.

In [4]:
summary = run_batch_preprocessing(
    raw_dir=RAW_DATA_DIR,
    processing_dir=PROCESSING_DATA_DIR,
    root_dir=ROOT_DIR,
    max_highlight_images=60,
    max_yolo_clips=300,
    val_ratio=0.2,
)

print("=== KẾT QUẢ XUẤT DỮ LIỆU TIỀN XỬ LÝ ===")
for k, v in summary.items():
    print(f"  {k:28s}: {v}")

📂 Đọc dữ liệu TuSimple từ: H:\FPT\4\CPV\project\CPV301\data\raw\TUSimple\train_set
📦 Tổng clip xử lý: 300 | val: 60 | train: 240


Chuyển đổi: 100%|██████████| 300/300 [00:42<00:00,  7.04clip/s]



✅ Hoàn tất. Thống kê:
   train:  240 ảnh |   917 polygon | bỏ qua 0
   valid:   60 ảnh |   225 polygon | bỏ qua 0

📂 Đầu ra: H:\FPT\4\CPV\project\CPV301\data\processing
👉 Tiếp theo: kiểm tra config/data.yaml (path=../data/processed) rồi chạy train.
=== KẾT QUẢ XUẤT DỮ LIỆU TIỀN XỬ LÝ ===
  processed_samples           : 60
  yolo_dataset_stats          : {'train': {'imgs': 240, 'polys': 917, 'skipped': 0}, 'valid': {'imgs': 60, 'polys': 225, 'skipped': 0}}
  csv_path                    : h:\FPT\4\CPV\project\CPV301\preprocessing_metrics.csv
  chart_path                  : h:\FPT\4\CPV\project\CPV301\preprocessing_pipeline_chart.png


In [5]:
# Đọc và kiểm tra bảng số liệu preprocessing_metrics.csv tại Root
csv_path = ROOT_DIR / "preprocessing_metrics.csv"
df_metrics = pd.read_csv(csv_path)
print(f"Đã lưu CSV tại Root: {csv_path} ({len(df_metrics)} dòng)")
display(df_metrics.head(10))

Đã lưu CSV tại Root: h:\FPT\4\CPV\project\CPV301\preprocessing_metrics.csv (60 dòng)


,sample_id,image_name,output_file,num_lanes_detected,uk_lane_type,edge_pixels_in_roi,color_pixels_in_roi,heading_error_rad,curvature,processing_time_ms
0,0,20.jpg,data/processing/highlighted/highlighted_00_20.jpg,2,uk_standard_left_lane,10807,0,0.29682,0.03475,238.47
1,1,curved_road.jpg,data/processing/highlighted/highlighted_01_cur...,2,uk_standard_left_lane,2552,10701,0.22012,0.13501,141.48
2,2,20.jpg,data/processing/highlighted/highlighted_02_20.jpg,2,uk_standard_left_lane,9371,9499,0.20072,0.10309,217.58
3,3,20.jpg,data/processing/highlighted/highlighted_03_20.jpg,1,uk_single_center_divider,36545,3983,-1.31402,0.02019,639.46
4,4,20.jpg,data/processing/highlighted/highlighted_04_20.jpg,2,uk_standard_left_lane,77294,25455,-0.77283,0.02902,698.44
5,5,20.jpg,data/processing/highlighted/highlighted_05_20.jpg,2,uk_standard_left_lane,153318,2354,0.07594,0.02163,1130.72
6,6,20.jpg,data/processing/highlighted/highlighted_06_20.jpg,2,uk_standard_left_lane,166334,6092,0.42174,0.30337,2591.00
7,7,20.jpg,data/processing/highlighted/highlighted_07_20.jpg,2,uk_standard_left_lane,147749,3609,0.56989,0.06688,1405.97
8,8,20.jpg,data/processing/highlighted/highlighted_08_20.jpg,2,uk_standard_left_lane,93576,9449,-0.22186,0.39049,1294.96
9,9,20.jpg,data/processing/highlighted/highlighted_09_20.jpg,2,uk_standard_left_lane,83701,7131,0.06516,0.26789,764.32
